# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Goal:** build two retrieval pipelines over the *same* chunked document and compare them.

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder retrieves Top 10 by cosine similarity | *same* Top 10 from Pipeline 1 |
| Stage 2 | – | Cross-encoder scores all 10 (query, chunk) pairs, re-sorts descending |
| Context for the LLM | Top 3 by cosine | Top 3 by cross-encoder score |
| Output | Answer 1 | Answer 2 (identical prompt template) |

**Document:** *Regulation (EU) 2016/679 – General Data Protection Regulation (GDPR)*, official EUR-Lex PDF (88 pages).
It is a good stress test for retrieval because:
- 173 recitals (non-binding explanations) paraphrase the 99 binding articles, so there are many near-duplicate distractors;
- rules are full of exceptions (*"unless…"*, *"shall not apply to…"*), negations and role-specific obligations (controller vs. processor vs. representative);
- many provisions refer to each other only by number (*"Articles 25 to 39"*).

**Models**
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2` (384-d embeddings)
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Vector store: FAISS `IndexFlatIP` over L2-normalised vectors (inner product = cosine similarity)
- LLM: Google Gemini (`gemini-2.5-flash`, temperature 0)

**Notebook layout**
1. Setup
2. Loading & cleaning the PDF
3. Chunking
4. Pipeline 1 – baseline retrieval
5. Pipeline 2 – cross-encoder re-ranking
6. Answer generation (shared prompt)
7. Failure mode analysis
8. Summary metrics
9. Latency profiling
10. Scaling experiment

## 1. Setup

All tunable settings are kept in one place. The Gemini API key is read from a local `.env` file
(`GEMINI_API_KEY=...`), which is git-ignored – see `.env.example`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os, re, json, time, random, hashlib, platform
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PyMuPDF, for PDF text + font information
import faiss                        # vector index
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)

# ---- configuration ----
PDF_PATH        = Path("data/GDPR_EU_2016_679.pdf")
RESULTS_DIR     = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CHUNK_MAX_WORDS = 150   # ~200 word-piece tokens, stays under MiniLM's 256-token limit
CHUNK_OVERLAP   = 30    # words carried over between consecutive chunks of the same article
TOP_K           = 10    # candidates retrieved by the bi-encoder
TOP_N           = 3     # chunks given to the LLM

load_dotenv()
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# fixed seeds so that everything except API latency is repeatable
random.seed(0); np.random.seed(0); torch.manual_seed(0)

print("Python  :", platform.python_version())
print("Machine :", platform.platform())
print("CPU     :", platform.processor() or platform.machine(), "|", os.cpu_count(), "logical cores")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")
print("Torch threads:", torch.get_num_threads())

Python  : 3.10.13
Machine : Linux-7.0.0-38-generic-x86_64-with-glibc2.39
CPU     : x86_64 | 8 logical cores
GPU     : none (CPU only)
Torch threads: 4


## 2. Loading & cleaning the PDF

Plain `get_text()` mixes three kinds of noise into the regulation text:
- the running footer on every page (`4.5.2016`, `L 119/12`, `Official Journal of the European Union`, `EN`),
- footnotes (`(1) OJ C 229, 31.7.2012, p. 90.`),
- chapter / section headings.

Instead of guessing with regexes alone, I use the **font information** PyMuPDF exposes:
body text is set in 9.6 pt, footnotes and the `EN` tag in 8.5 pt, article headings (`Article 17`) in italic and
article titles in bold. Each line is kept together with its page number and bold/italic flags.

In [2]:
FOOTER_RE  = re.compile(r"^(\d{1,2}\.\d{1,2}\.\d{4}|L \d+/\d+|Official Journal of the European Union)$")
BODY_SIZE  = 9.0          # anything smaller than this is a footnote / page tag
BOLD, ITALIC = 16, 2      # PyMuPDF span flag bits


def load_lines(path):
    # Return every body-text line as {page, text, bold, italic}.
    lines = []
    for page_no, page in enumerate(pymupdf.open(path), start=1):
        for block in page.get_text("dict")["blocks"]:
            for line in block.get("lines", []):
                spans = [s for s in line["spans"] if s["size"] >= BODY_SIZE]   # drop footnote-sized spans
                text = "".join(s["text"] for s in spans).strip()
                if not text or FOOTER_RE.match(text):                          # drop the running footer
                    continue
                flags = spans[0]["flags"]
                lines.append({"page": page_no, "text": text,
                              "bold": bool(flags & BOLD), "italic": bool(flags & ITALIC)})
    return lines


lines = load_lines(PDF_PATH)
print(f"{pymupdf.open(PDF_PATH).page_count} pages -> {len(lines)} body-text lines")
pd.DataFrame(lines[:8])

88 pages -> 4124 body-text lines


,page,text,bold,italic
0,1,I,False,False
1,1,(Legislative acts),False,True
2,1,REGULATIONS,False,False
3,1,REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL,True,False
4,1,of 27 April 2016,True,False
5,1,on the protection of natural persons with regard to the processing of personal data and on the free,True,False
6,1,"movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation)",True,False
7,1,"Having regard to the Treaty on the Functioning of the European Union, and in particular Article 16 thereof,",False,False


### 2.1 Splitting into recitals and articles

The regulation has two parts:
- **Recitals** `(1)` … `(173)` – the "whereas" explanations (pages 1–31);
- **Articles** `Article 1` … `Article 99` – the binding rules (pages 32–88).

I group lines into one *unit* per recital / article and keep the article title as metadata
(e.g. `Article 33 - Notification of a personal data breach to the supervisory authority`).
A recital number only starts a new recital if it is the next number in sequence, so inline
references like "(4)" do not break the text. Words hyphenated across line breaks are re-joined.

In [3]:
ARTICLE_RE = re.compile(r"^Article (\d+)$")
RECITAL_RE = re.compile(r"^\((\d+)\)\s*(.*)$")


def to_units(lines):
    units, cur = [], None
    in_articles, title_open, next_recital = False, False, 1

    def flush():
        if cur and cur["text"].strip():
            units.append(cur)

    for ln in lines:
        t = ln["text"]
        # the first italic 'Article 1' heading marks the end of the recitals
        if not in_articles and ARTICLE_RE.match(t) and ln["italic"]:
            in_articles = True

        if not in_articles:
            if ln["bold"]:
                continue                                    # title block / headings
            m = RECITAL_RE.match(t)
            if m and int(m.group(1)) == next_recital:
                flush()
                cur = {"section": f"Recital ({next_recital})", "page": ln["page"], "text": m.group(2)}
                next_recital += 1
            elif cur is not None:
                cur["text"] += " " + t
            continue

        m = ARTICLE_RE.match(t)
        if m and ln["italic"]:                              # new article heading
            flush()
            cur = {"section": f"Article {m.group(1)}", "page": ln["page"], "text": ""}
            title_open = True
            continue
        if ln["bold"]:
            if cur is not None and title_open:              # article title, can wrap over 2-3 lines
                sep = " - " if " - " not in cur["section"] else " "
                cur["section"] += sep + t
            continue                                        # chapter / section headings are dropped
        title_open = False
        if cur is not None:
            cur["text"] += " " + t
    flush()

    for u in units:
        txt = re.sub(r"(\w)- (\w)", r"\1\2", u["text"])     # re-join hyphenated words
        u["text"] = re.sub(r"\s+", " ", txt).strip()
    return units


units = to_units(lines)
n_rec = sum(u["section"].startswith("Recital") for u in units)
n_art = sum(u["section"].startswith("Article") for u in units)
print(f"{n_rec} recitals, {n_art} articles")
assert n_rec == 173 and n_art == 99, "structure parsing went wrong"
pd.DataFrame(units)[["section", "page", "text"]].iloc[[0, 172, 173, 205]]

173 recitals, 99 articles


,section,page,text
0,Recital (1),1,The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1...
172,Recital (173),31,This Regulation should apply to all matters concerning the protection of fundamental rights and freedoms vis-àvis th...
173,Article 1 - Subject-matter and objectives,32,1. This Regulation lays down rules relating to the protection of natural persons with regard to the processing of pe...
205,Article 33 - Notification of a personal data breach to the supervisory authority,52,"1. In the case of a personal data breach, the controller shall without undue delay and, where feasible, not later th..."


## 3. Chunking

Each recital/article is split into chunks of at most **150 words** with a **30-word overlap**, cutting on
sentence (or `;` / `:`) boundaries so that numbered paragraphs stay intact. Chunks never cross an article
boundary, so every chunk carries a clean `section` label and page number – useful for checking the LLM's
citations later.

150 words is chosen to fit the bi-encoder: `all-MiniLM-L6-v2` truncates input at 256 word-piece tokens, and
anything past that limit would silently be ignored during embedding. The check below confirms no chunk is
truncated.

This list of chunks is built **once** and is used by both pipelines.

In [4]:
SENT_SPLIT_RE = re.compile(r"(?<=[.;:])\s+(?=[A-Z(0-9])")


def chunk_units(units, max_words=CHUNK_MAX_WORDS, overlap=CHUNK_OVERLAP):
    chunks = []
    for u in units:
        # sentence-like pieces; very long sentences are hard-split with overlap
        pieces = []
        for sent in SENT_SPLIT_RE.split(u["text"]):
            words = sent.split()
            while len(words) > max_words:
                pieces.append(" ".join(words[:max_words]))
                words = words[max_words - overlap:]
            if words:
                pieces.append(" ".join(words))

        # greedily pack pieces into chunks, carrying the last `overlap` words forward
        buf = []
        for piece in pieces:
            if buf and len(" ".join(buf + [piece]).split()) > max_words:
                chunks.append({"section": u["section"], "page": u["page"], "text": " ".join(buf)})
                buf = [" ".join(" ".join(buf).split()[-overlap:])]
            buf.append(piece)
        if buf:
            chunks.append({"section": u["section"], "page": u["page"], "text": " ".join(buf)})

    for i, c in enumerate(chunks):
        c["chunk_id"] = i
    return chunks


chunks = chunk_units(units)          # <- the single shared corpus for both pipelines
chunk_texts = [c["text"] for c in chunks]
chunks_df = pd.DataFrame(chunks).set_index("chunk_id")

word_counts = chunks_df["text"].str.split().str.len()
print(f"{len(chunks)} chunks "
      f"({chunks_df.section.str.startswith('Recital').sum()} from recitals, "
      f"{chunks_df.section.str.startswith('Article').sum()} from articles)")
word_counts.describe().round(1).to_frame("words per chunk").T

565 chunks (258 from recitals, 307 from articles)


,count,mean,std,min,25%,50%,75%,max
words per chunk,565.0,109.5,32.9,22.0,85.0,119.0,137.0,164.0


In [5]:
# Make sure no chunk exceeds the bi-encoder's 256 token window (otherwise its tail is never embedded)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BI_ENCODER_NAME)
token_counts = pd.Series([len(tok(t)["input_ids"]) for t in chunk_texts])
print("tokens per chunk: mean %.0f | max %d | chunks over 256 tokens: %d"
      % (token_counts.mean(), token_counts.max(), (token_counts > 256).sum()))

# a few sample chunks
chunks_df.loc[[0, 323, 366]]

tokens per chunk: mean 128 | max 196 | chunks over 256 tokens: 0


,section,page,text
chunk_id,,,
0,Recital (1),1,The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1...
323,Article 17 - Right to erasure (‘right to be forgotten’),43,1. The data subject shall have the right to obtain from the controller the erasure of personal data concerning him o...
366,Article 33 - Notification of a personal data breach to the supervisory authority,52,"1. In the case of a personal data breach, the controller shall without undue delay and, where feasible, not later th..."
